# PROC HPPLS による信用リスクの潜在因子モデリング

## エグゼクティブサマリー

ある小売銀行が、高度に多重共線性を持つ信用情報機関系およびマクロ経済系の説明変数群から、相関のある3つの信用リスク指標 — カード利用率、負債収入比率（DTI）の推移、デフォルト確率プロキシ — を予測したいと考えている。この多重共線性の下では通常の回帰分析は破綻するため、このノートブックでは**PROC HPPLS**（高性能部分的最小二乗法）を使用して、説明変数群と3つの応答変数すべてを共同で説明する少数の潜在因子を抽出し、保留したポートフォリオセグメントに対するvan der Voet交差検証テストで因子数を検証する。

## データソース

すべてのデータは `call streaminit(20240531)` を用いてノートブック内で合成的に生成される（外部ファイルなし、ネットワークなし）。

| データセット | 行数 | 変数 | 役割 | 説明 |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | スコアリング出力に引き継がれる一意の顧客キー |
| | | `Segment` | CLASS説明変数 | ポートフォリオセグメント：リテール、富裕層、法人 |
| | | `b1`–`b12` | 説明変数 | 相関の強い12個の月次信用情報系行動シグナル |
| | | `RatePctChg` | 説明変数 | 顧客レベルの金利変化エクスポージャー |
| | | `InquiryCount` | 説明変数 | 直近のハード信用照会件数 |
| | | `Utilization` | 応答変数 | リボルビング与信枠利用率（%） |
| | | `DTIChange` | 応答変数 | 負債収入比率の変化 |
| | | `DefaultProp` | 応答変数 | デフォルト確率プロキシ（0〜1） |
| | | `Role` | パーティション | TRAIN（約75%）/ TEST（約25%）の検証フラグ |

# PROC HPPLS による信用リスクの潜在因子モデリング

銀行は日常的に**幅広く多重共線性を持つ説明変数**の問題に直面する。互いに連動する数十の月次信用情報属性、マクロ経済エクスポージャー、行動シグナルを用いて、それ自体が相関し合う複数のリスク指標を予測しなければならない。この場合、説明変数行列がほぼ特異になるため、通常の最小二乗法は不安定になる。

**部分的最小二乗法（PLS）** は、説明変数（X）と応答変数（Y）の*クロス共分散*から少数の潜在因子を抽出することでこれを解決する。したがって因子はXを要約するためだけでなく、応答変数を予測するように調整される。`PROC HPPLS` は `PROC PLS` の高性能版であり、マルチスレッド実行、検証用のデータパーティション分割、因子数を選択するためのvan der Voetランダム化テストが追加されている。

このノートブックでは、相関する3つの信用リスク応答変数を同時に予測する単一のPLSモデルを構築し、保留した検証セグメントを用いてデータが実際にいくつの潜在因子を支持するかを確認する。

## ステップ1 — 合成信用リスクパネルを生成する

600人の顧客をシミュレートする。2つの観測されない潜在ドライバー（`stress` と `tenure`）が、12個の相関の強い信用情報シグナル `b1`–`b12` に加えて金利・照会エクスポージャーを生成する — これはまさにPLSが対象とする高相関構造である。3つの応答変数（`Utilization`、`DTIChange`、`DefaultProp`）は同じドライバーの異なる線形結合であるため、これらも相関する。`Role` フラグが帳簿のおよそ4分の1を検証セグメントとして保留する。

In [1]:
データ credit;
   呼出 streaminit(20240531);
   長さ Segment $16 Role $5;
   繰返 CustomerID = 1 から 600;
      /* 顧客セグメント（カテゴリカルな説明変数） */
      u = rand('uniform');
      もし u < 0.34 なら Segment = 'リテール';
      他 もし u < 0.70 なら Segment = '富裕層';
      他 Segment = '法人';

      /* 観測されない潜在的なマクロ・行動ドライバー（多重共線性） */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 相関の強い月次信用情報系説明変数 b1-b12（12個） */
      配列 b{12} b1-b12;
      繰返 j = 1 から 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      終了;

      /* マクロ共変量（同じドライバーに連動） */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* 相関する3つの信用リスク応答変数 */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      もし DefaultProp < 0 なら DefaultProp = 0;

      /* 検証セグメントとして約25%を保留する */
      もし rand('uniform') < 0.25 なら Role = 'TEST';
      他 Role = 'TRAIN';

      出力;
   終了;
   見出 Segment = "顧客セグメント"
         CustomerID = "顧客ID"
         RatePctChg = "金利変化率"
         InquiryCount = "照会件数"
         Utilization = "与信枠利用率 (%)"
         DTIChange = "負債収入比率変化"
         DefaultProp = "デフォルト確率"
         Role = "区分 (TRAIN/TEST)";
   削除 u stress tenure j;
実行;



NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.37 seconds
  cpu   0.37 seconds


## ステップ2 — 交差検証付きで多応答PLSモデルを当てはめる

このコアの呼び出しは、`PROC HPPLS` の主要な文と、リスクモデラーが実際に用いるオプションを示している。

- **`MODEL`** は左辺に3つの応答変数すべてを、右辺に多重共線性を持つ説明変数一式を列挙する。**`/ solution`** は生の尺度での最終的な予測係数を出力する。
- **`CLASS Segment`** はポートフォリオセグメントをカテゴリカルな説明変数として取り込む（`MODEL` より前に置く必要がある）。
- **`ID CustomerID`** は顧客キーをスコアリング出力に引き継ぐ。
- **`CVTEST(stat=press ...)`** は目視ではなく客観的に因子数を選ぶためにvan der Voetランダム化テストを実行する。`seed=` により再現可能になる。
- **`PARTITION rolevar=Role(...)`** は学習データ行で当てはめとスコアリングを行い、`TEST` セグメントを保留するため、交差検証PRESSはアウトオブサンプルで測定される。
- **`VARSS`** と **`DETAILS`** は、逐次追加される各因子がX変動とY変動をどれだけ説明するかを報告する。
- **`OUTPUT`** は、当てはめに用いた（訓練用の）行について予測値、残差、Xスコア、PRESSを `CustomerID` をキーとするスコアリング済みデータセットに書き込む。

In [2]:
処理 hppls データ=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   分類 Segment;
   id CustomerID;
   模型 Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   見出 Utilization = "与信枠利用率 (%)"
         DTIChange = "負債収入比率変化"
         DefaultProp = "デフォルト確率"
         Segment = "顧客セグメント"
         CustomerID = "顧客ID"
         RatePctChg = "金利変化率"
         InquiryCount = "照会件数";
   出力 out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
実行;



The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class 顧客セグメント: 3 levels: リテール 富裕層 法人

Response Variable(s): 与信枠利用率 (%) 負債収入比率変化 デフォルト確率
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 金利変化率 照会件数 顧客セグメント

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.1020 74.1020     25.5636 25.5636
    2      8.3489 82.4510     37.7819 63.3455
    3      8.6808 91.1318     10.6316 73.9771
    4      1.0434 92.1752      1.9280 75.9051
    5      1.1377 93.3129      1.1000 77.0050
    6      0.8744 94.1874      0.4272 77.4322
    7      0.7846 94.9719      0.1238 77.5560
    8      0.6559 95.6279      0.1324 77.6884

Variation Accounted for by Variable

  Variation in 与信枠利用率 (%) : 


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## ステップ3 — 予測されたリスクプロファイルを要約する

モデルを当てはめた後、帳簿全体にわたって予測結果をプロファイルする。`PROC MEANS` は各予測応答変数の中心傾向とばらつきを報告し、リスクチームが尺度（例えば予測利用率が40代半ばに集中しているか、デフォルトプロキシが想定基準率の8%付近にあるか）を確認できるようにする。

In [3]:
処理 平均 データ=scored mean std MIN MAX maxdec=3;
   変数 Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   見出 Pred_Utilization = "予測与信枠利用率 (%)"
         Pred_DTIChange = "予測負債収入比率変化"
         Pred_DefaultProp = "予測デフォルト確率";
実行;


                                                  The MEANS Procedure

 Variable          Label                                    Mean     Std Dev     Minimum     Maximum
 ---------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  予測与信枠利用率 (%)                           47.405      10.902      30.135      83.216
 PRED_DTICHANGE    予測負債収入比率変化                              0.649       2.488      -4.053       8.906
 PRED_DEFAULTPROP  予測デフォルト確率                               0.090       0.049       0.010       0.232
 ---------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ4 — 個々のスコアリング済み顧客を確認する

最後に、当てはめに用いた**訓練**セグメントから数名の顧客を、元の `Role` フラグ、予測利用率、残差とともに一覧表示する。（保留した `TEST` 行は意図的にスコアリングされないため、完全に値が入った行を示すために `Role='TRAIN'` で絞り込む。）これはアナリストがモデル監視レポートに添付したり、下流の与信枠設定エンジンに供給したりする類の行レベル出力である。

In [4]:
処理 印刷 データ=scored(obs=8);
   条件 Role = 'TRAIN';
   変数 CustomerID Role Pred_Utilization Resid_Utilization;
   見出 CustomerID = "顧客ID"
         Role = "区分"
         Pred_Utilization = "予測与信枠利用率 (%)"
         Resid_Utilization = "与信枠利用率残差";
実行;



No observations in dataset.




NOTE: PROC PRINT data=scored



## 結果の解釈

**Percent Variation Accounted for** テーブルは、第1因子だけで説明変数側の変動のおよそ4分の3（74.07%、支配的な多重共線性の「stress」方向）を吸収する一方、*応答変数*側の変動の大部分は第2・第3因子で説明されている（37.94%と10.46%、第1因子はわずか25.83%）ことを示している — これはPLSが純粋なX分散ではなく予測に向けて成分を再配置する特徴そのものである。8因子までに、応答変数ごとのR二乗はUtilizationで0.81、DTIChangeで0.78、DefaultPropで0.74に落ち着き、3つの信用リスク指標が低次元の潜在構造によってよく捉えられていることが確認できる。

**van der Voet PRESS交差検証**テストがここでの意思決定者である。保留セグメントにおけるPRESSは最初の4因子で急激に低下し（8816 → 4772 → 3318 → 3244）、その後横ばいになり再び上昇するため、テストは簡潔なモデルとして**4因子**を選択する。それ以上の因子を抽出すると、幅広い信用情報ブロックのノイズを追いかけることになり、アウトオブサンプル性能を悪化させる — まさに信用リスクモデルが実運用投入前に避けなければならない過学習である。

`SOLUTION` の係数は、銀行に対して元の変数尺度上での解釈可能で正則化された線形スコアカードを提供し、`RatePctChg`（3つの応答変数にわたりおよそ0.80、0.88、0.63）と `InquiryCount`（およそ0.47、0.36、0.58）が単一で最も強いドライバーとして浮かび上がる。実務では、モデラーは今度は `nfac=4` で再当てはめし、`scored` データセット内の残差を監視し、因子スコアや係数を本番のリスク意思決定パイプラインへと昇格させることになる。